## A notebook to analyze groups of croparrays  

### Read in, combine, and making measurements 

In [1]:
import croparray as ca
from pathlib import Path
%gui qt

In [19]:
# ============================
# USER-DEFINED PARAMETERS
# ============================

# Files & Directories
DATA_DIR_1 = Path(r"\\129.82.125.64\TSnas2\SharedInternal\RiboJamProject\20260325_ZAK_KO_ST_KDM5B\noDox\HT\CropArrays")
DATA_DIR_2 = Path(r"\\129.82.125.64\TSnas2\SharedInternal\RiboJamProject\20260325_ZAK_KO_ST_KDM5B\wDox\HT\CropArrays")
LABELS = ['-Zak','+Zak']
OUTPUT = Path(r"\\129.82.125.64\TSnas2\SharedInternal\RiboJamProject\20260325_ZAK_KO_ST_KDM5B") 

nc_files_1 = sorted(DATA_DIR_1.glob("*.nc"))
nc_files_2 = sorted(DATA_DIR_2.glob("*.nc"))

# Measurement parameters
REF_CH = 0  # Optional reference channel for best_z_proj (if use_zc=False, i.e. when Z not tracked)
DISK_R = 5  # Radius of disk in pixels to measure signal
DISK_BG = 7 # Radius of single pixel ring around disk to measure background 
ROLL_N = 1  # Rolling z-average for signal measurement.

In [9]:
my_ca = ca.build.open_measure_concat(
    groups=[nc_files_1, nc_files_2],
    dims=["exp", "fov"],
    labels=[["-Zak", "+Zak"], None],
    measure_kwargs=dict(ref_ch=REF_CH, disk_r=DISK_R, disk_bg=DISK_BG, roll_n=ROLL_N,drop_int=True, # drop full z-stack to save memory, keeping best_z_proj
    ),
    open_as = "croparray",  # can be croparray or trackarray 
    join="outer",
)

[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.


In [10]:
# Create binary masks from "best_z" so can measure morphological properties. 
my_ca.ops.apply(
    ca.tools.binarize_crop_manual,
    channels=[0],               # Channel used to generate mask
    source="best_z",            # Image layer to threshold (typically z-projected signal)
    out_name="ch{ch}_mask",     # Output mask variable name (→ "ch0_mask")

    # Recommended parameters forwarded directly to binarize_crop_manual(...)
    func_kwargs=dict(
        q=0.45,                 # Threshold at 45% after intensity normalization
        q_range=(0.02, 0.999),  # Normalize intensities using 2%–99.9% quantile range
        q_positive_only=True,   # Ignore negative pixels (important for best_z)
        close_px=1,             # Morphological closing (bridge small gaps)
        smooth_px=0,            # Light smoothing of mask edges
        fill_holes=False,       # Do not fill internal holes
        return_uint8=True,      # Store mask as uint8 (0/1)
        morph_px=0,             # Apply a dilation of final mask (morph_pix > 0; <0 for erosion)
    ),
);

In [11]:
# Check masks with napari
viewer, layers = my_ca.napari.montage_viewer(
    row="n",
    col="t",
    show=("best_z", "ch0_mask"),
    #ch=[0, 1, 2],  # RGB
    colormaps={"ch0_mask": "magenta","best_z": "green"},
    image_contrast=[0.5,99.5],
    show_tile_text=False,
)

In [12]:
# Make measurements of desired properties 
# See https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops
my_ca.measure.mask_props(source ="ch0_mask", props=['eccentricity','major_axis_length_px','minor_axis_length_px']);

In [20]:
# =======================================
# SAVE YOUR WORK TO A NEW CROP/TRACKARRAY
# =======================================

my_file = ca.io.save_croparray(my_ca, output_dir=OUTPUT, ext='.nc')
my_file

C:\Users\StasLab\Github\croparray\croparray\io.py:467: SerializationWarning: saving variable xc_pix with floating point data as an integer dtype without any _FillValue to use for NaNs
  ds_to_save.to_netcdf(str(out_path), **kwargs)
c:\Users\StasLab\anaconda3\envs\croparray_env\lib\site-packages\xarray\core\duck_array_ops.py:253: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)
C:\Users\StasLab\Github\croparray\croparray\io.py:467: SerializationWarning: saving variable yc_pix with floating point data as an integer dtype without any _FillValue to use for NaNs
  ds_to_save.to_netcdf(str(out_path), **kwargs)
C:\Users\StasLab\Github\croparray\croparray\io.py:467: SerializationWarning: saving variable zc_pix with floating point data as an integer dtype without any _FillValue to use for NaNs
  ds_to_save.to_netcdf(str(out_path), **kwargs)
C:\Users\StasLab\Github\croparray\croparray\io.py:467: SerializationWarning: saving variable xc_pad with floating poi

WindowsPath('//129.82.125.64/TSnas2/SharedInternal/RiboJamProject/20260325_ZAK_KO_ST_KDM5B/PB_MAX_02__12exp_fov.nc')

In [17]:
# check your saved data; note concatenation history is saved
temp = ca.tools.open_croparray(my_file)
temp.ds

[CropArray] No sidecars found.


<xarray.Dataset> Size: 513MB
Dimensions:                         (n: 229, t: 46, y: 21, x: 21, fov: 6,
                                     exp: 2, ch: 1, z: 1)
Coordinates:
  * n                               (n) int16 458B 0 1 2 3 4 ... 225 226 227 228
  * t                               (t) int32 184B 0 1 2 3 4 ... 41 42 43 44 45
  * y                               (y) float64 168B -1.48 -1.332 ... 1.332 1.48
  * x                               (x) float64 168B -1.48 -1.332 ... 1.332 1.48
  * fov                             (fov) int32 24B 0 1 2 3 4 5
  * exp                             (exp) object 16B '-Zak' '+Zak'
  * z                               (z) float64 8B 0.0
  * ch                              (ch) int32 4B 0
Data variables: (12/25)
    ch0_mask                        (exp, fov, n, t, x, y) int8 56MB ...
    xc                              (exp, fov, n, t, ch) float32 506kB ...
    yc                              (exp, fov, n, t, ch) float32 506kB ...
    zc                              (exp, fov, n, t, ch) float32 506kB ...
    xc_pix                          (exp, fov, n, t, ch) int16 253kB ...
    yc_pix                          (exp, fov, n, t, ch) int16 253kB ...
    ...                              ...
    ch0_mask__eccentricity          (exp, fov, n, t) float64 1MB ...
    ch0_mask__major_axis_length_px  (exp, fov, n, t) float64 1MB ...
    ch0_mask__minor_axis_length_px  (exp, fov, n, t) float64 1MB ...
    fov_name                        (exp, fov, n) object 22kB ...
    best_z                          (exp, fov, n, ch, t, y, x) float64 446MB ...
    signal                          (exp, fov, n, ch, t) float64 1MB ...
Attributes: (12/16)
    name:                      PB_MAX_02 +11 (concat[12] exp×fov)
    date:                      2026-03-25
    xy_pad:                    10
    z_pad:                     1
    dx:                        0.148
    dy:                        0.148
    ...                        ...
    signal_units:              a.u.
    croparray_schema_version:  2.0
    notes:                     A crop array in KO Zak U2OS cells with tet-ind...
    provenance_json:           {\n  "timestamp": "2026-03-27T19:02:01",\n  "d...
    concat_meta_json:          {"base_name": "PB_MAX_02", "dims": ["exp", "fo...
    filename:                  PB_MAX_02__12exp_fov.nc

### Auto and/or manual filtering of saved croparray

In [26]:
import croparray as ca
from pathlib import Path
%gui qt

In [27]:
# ============================
# USER-DEFINED PARAMETERS
# ============================

INPUT_FILE = Path(r"\\129.82.125.64\TSnas2\SharedInternal\RiboJamProject\20260325_ZAK_KO_ST_KDM5B\PB_MAX_02__12exp_fov.nc")

In [28]:
# Need to open this way to get proper doc strings:
from croparray.trackarray.object import CropArray
my_ca: CropArray = ca.io.open_croparray(INPUT_FILE, as_object=True)

[CropArray] Found 1 sidecar(s).
[CropArray] Checking sidecar: \\129.82.125.64\TSnas2\SharedInternal\RiboJamProject\20260325_ZAK_KO_ST_KDM5B\PB_MAX_02__12exp_fov__manual__bad.nc
[CropArray] Merging sidecar \\129.82.125.64\TSnas2\SharedInternal\RiboJamProject\20260325_ZAK_KO_ST_KDM5B\PB_MAX_02__12exp_fov__manual__bad.nc with vars: ['bad']
[CropArray] Merging sidecars into CropArray dataset.


In [ ]:
# Create a layer that automatically finds bad spots that have mask area bigger thanb 2x median and have a mask center of mask that is more than 4 pixels away from the signal center. This happens when the signal is very dim and the mask is basically random noise, which can cause bad measurements. This will create a new boolean layer called "auto_bad" that you can use to filter out these bad spots in your analysis. You can adjust the parameters as needed for your specific dataset.
my_ca.measure.auto_bad(source='ch1_mask', area_factor=2.0, max_dist_px=4, exclude_empty=False, out_name='auto_bad', save_sidecar=True, output_dir=INPUT_FILE.parent);

In [ ]:
# Check the new "auto_bad" layer with napari; adjust parameters as needed to capture bad spots without being too stringent. You can also compare with your existing "bad" layer to see if this new method is capturing additional bad spots that were missed before.
my_ca.view.montage_viewer(row='n', col='t',ch=1, show=('best_z', 'auto_bad', 'ch1_mask'), colormaps={'best_z': 'green', 'auto_bad': 'yellow','ch1_mask':'red'})

In [ ]:
# Quick and user-friendly manual filtering of tracks with napari 
# Create and save filter names (e.g. a 'bad' filter to remove noisy crops/tracks)
# Can also create filters like 'toi' to highlight tracks of interest  
filter_name = "bad"
viewer, layers, ft = my_ca.napari.manual_filter_montage(
    row="n",  # can be 'track_id' for trackarrays or 'n' for croparrays               
    col="t",  
    filter_name=filter_name,
    show=("best_z", "ch0_mask"),
    ch=0,
    colormaps={"ch0_mask":"magenta", "best_z":"green", filter_name:"yellow"},
    output_dir= INPUT_FILE.parent,
    show_tile_text=False, # adds layer w/ crop coords; takes memory,
    show_click_info=True, # useful for debugging
)

INFO: Saved manual filter sidecar:
\\129.82.125.64\TSnas2\SharedInternal\RiboJamProject\20260325_ZAK_KO_ST_KDM5B\PB_MAX_02__12exp_fov__manual__bad.nc
INFO: Saved manual filter sidecar:
\\129.82.125.64\TSnas2\SharedInternal\RiboJamProject\20260325_ZAK_KO_ST_KDM5B\PB_MAX_02__12exp_fov__manual__bad.nc


In [ ]:
# # Test your manual filter:
my_ta.where(my_ta.ds[filter_name] == 0).napari.montage_viewer(row="track_id", col="t",show=("best_z","ch0_mask"),ch=0,show_tile_text=False)